# egfr40 quickstart

Score the 3-year risk of a ≥40% eGFR decline or kidney failure for a small, made-up
cohort, then see what drives each individual's risk.

## 1. A small cohort

Five made-up individuals: one in each of the model's four strata (diabetes or not,
eGFR ≥60 or <60), plus one with a missing blood pressure.

In [ ]:
import pandas as pd
from egfr40 import add_egfr40_risk_col, egfr40_risk

# One row per individual; column names match add_egfr40_risk_col's defaults
cohort = pd.DataFrame({
    "individual": ["A", "B", "C", "D", "E"],
    "age":      [50, 58, 72, 70, 64],
    "sex":      ["female", "male", "male", "male", "female"],
    "eGFR":     [95, 82, 42, 40, 75],
    "uACR":     [8, 40, 150, 150, 25],          # mg/g
    "sbp":      [120, 138, 140, 140, None],     # individual E has no BP reading
    "bp_meds":  ["no", "yes", "yes", "yes", "no"],
    "hf":       [0, 0, 0, 0, 0],
    "chd":      [0, 0, 1, 1, 0],
    "afib":     [0, 0, 1, 1, 0],
    "smoking":  ["never", "current", "former", "former", "never"],
    "bmi":      [25, 31, 28, 28, 27],
    "dm":       [0, 1, 0, 1, 0],
    "hba1c":    [None, 8.0, None, 7.0, None],   # % ; only needed when dm = 1
    "oral_dm":  [0, 1, 0, 0, 0],
    "insulin":  [0, 0, 0, 0, 0],
})
cohort

## 2. Add a risk column for every individual

`add_egfr40_risk_col` scores each row with its own stratum and adds `egfr40_3year`.
Individual E gets `NaN` because a required input (systolic BP) is missing.

In [ ]:
scored = add_egfr40_risk_col(cohort)

# Label each individual's stratum to see which model was used
scored["stratum"] = (
    scored["dm"].map({0: "No DM", 1: "DM"})
    + ", eGFR "
    + scored["eGFR"].ge(60).map({True: ">=60", False: "<60"})
)
scored["risk_%"] = (scored["egfr40_3year"] * 100).round(1)

scored[["individual", "stratum", "eGFR", "uACR", "egfr40_3year", "risk_%"]]

## 3. What drives one individual's risk

`explain=True` breaks individual D's risk into each predictor's contribution to the
log-odds. Positive values raise risk; the largest drivers come first.

In [ ]:
out = egfr40_risk(
    age=70, male=1, egfr=40, uacr=150, sbp=140, bp_meds=1,
    hf=0, chd=1, afib=1, smoking="former", bmi=28,
    dm=True, hba1c=7, explain=True,
)
print(out["stratum"], f"risk = {out['risk']:.1%}")

pd.Series(out["contributions"], name="log-odds contribution").round(3).to_frame()

## 4. Top drivers for every individual

Apply `explain=True` row by row and keep each individual's three largest drivers.
Rows with missing inputs are skipped.

In [ ]:
def top_drivers(row, n=3):
    """Return the n largest contributions for one complete row."""
    out = egfr40_risk(
        age=row.age, male=int(row.sex == "male"), egfr=row.eGFR, uacr=row.uACR,
        sbp=row.sbp, bp_meds=int(row.bp_meds == "yes"), hf=row.hf, chd=row.chd,
        afib=row.afib, smoking=row.smoking, bmi=row.bmi, dm=bool(row.dm),
        hba1c=row.hba1c if row.dm else None, oral_dm=row.oral_dm,
        insulin=row.insulin, explain=True,
    )
    top = list(out["contributions"].items())[:n]
    return ", ".join(f"{k} {v:+.2f}" for k, v in top)

complete = scored.dropna(subset=["egfr40_3year"])
complete.assign(top_drivers=complete.apply(top_drivers, axis=1))[
    ["individual", "stratum", "risk_%", "top_drivers"]
]